In [ ]:
# --- Act 0: Setup (hidden) ---
import warnings
warnings.filterwarnings('ignore')

import html
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from textwrap import shorten

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats
from IPython.display import HTML, display, Markdown

# --- Path resolution ---
_here = Path('.').resolve()
if (_here / 'projects').exists():
    REPO_ROOT = _here
elif (_here.parent / 'projects').exists():
    REPO_ROOT = _here.parent
else:
    raise FileNotFoundError(
        "Cannot find 'projects/' directory. "
        "Run this notebook from the repository root or the notebooks/ directory."
    )
CYCLE = REPO_ROOT / 'projects' / 'owasp-llm' / 'cycles' / '2026'

# --- Plotly config ---
pio.renderers.default = "notebook+png"
pio.templates.default = "plotly_white"

# --- Seaborn / matplotlib theme ---
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

# --- Color palette ---
ENTRY_IDS = [
    'LLM01', 'LLM02', 'LLM03', 'LLM04', 'LLM05',
    'LLM06', 'LLM07', 'LLM08', 'LLM09', 'LLM10',
    'NEW-ITSCD', 'NEW-MA', 'NEW-MSDA', 'NEW-MTIE', 'NEW-PMP', 'NEW-WLA',
    'ROLL-CFAS', 'ROLL-CMSB', 'ROLL-LAPTF', 'ROLL-SICG',
]
FRAME_BLIND = {'LLM04', 'LLM08', 'LLM10'}

_incumbent_blues = sns.color_palette('mako', n_colors=12)
_new_oranges = sns.color_palette('flare', n_colors=8)
_roll_purples = sns.color_palette('crest', n_colors=6)
ENTRY_COLORS = {}
_ib = 0; _in = 0; _ir = 0
for eid in ENTRY_IDS:
    if eid in FRAME_BLIND:
        ENTRY_COLORS[eid] = '#999999'
    elif eid.startswith('LLM'):
        ENTRY_COLORS[eid] = _incumbent_blues[_ib % len(_incumbent_blues)]
        _ib += 1
    elif eid.startswith('NEW'):
        ENTRY_COLORS[eid] = _new_oranges[_in % len(_new_oranges)]
        _in += 1
    elif eid.startswith('ROLL'):
        ENTRY_COLORS[eid] = _roll_purples[_ir % len(_roll_purples)]
        _ir += 1

# --- Deep-dive sidebar helper ---
def sidebar(title, content):
    """Render a collapsible deep-dive sidebar."""
    return HTML(
        f'<details style="margin: 1em 0; padding: 0.5em; '
        f'border-left: 3px solid #4a86c8; background: #f8f9fa;">'
        f'<summary style="cursor: pointer; font-weight: bold; '
        f'color: #2c5282;">{title}</summary>'
        f'<div style="margin-top: 0.8em; line-height: 1.6;">{content}</div>'
        f'</details>'
    )

def callout_box(text, border_color='#e53e3e', bg_color='#fff5f5'):
    """Render an always-visible boxed callout (not collapsed)."""
    return HTML(
        f'<div style="margin: 1em 0; padding: 1em; border-left: 4px solid {border_color}; '
        f'background: {bg_color}; line-height: 1.6;">{text}</div>'
    )

# --- Load all data ---
DATA = {}

with open(CYCLE / 'prereg' / 'rubric.json') as f:
    DATA['rubric'] = json.load(f)

with open(CYCLE / 'classify' / 'labeled_incidents_multimodel.json') as f:
    DATA['incidents'] = json.load(f)

prelabels = []
with open(CYCLE / 'calibration' / 'llm_prelabels.jsonl') as f:
    for line in f:
        prelabels.append(json.loads(line))
DATA['prelabels'] = prelabels

goldset = []
with open(CYCLE / 'calibration' / 'adjudicated_goldset.jsonl') as f:
    for line in f:
        goldset.append(json.loads(line))
DATA['goldset'] = goldset

precision_verif = []
with open(CYCLE / 'calibration' / 'precision_verification.jsonl') as f:
    for line in f:
        precision_verif.append(json.loads(line))
DATA['precision_verification'] = precision_verif

with open(CYCLE / 'calibration' / 'posteriors.json') as f:
    DATA['posteriors'] = json.load(f)

with open(CYCLE / 'calibration' / 'diagnostic.json') as f:
    DATA['diagnostic'] = json.load(f)

with open(CYCLE / 'infer' / 'inference_summary.json') as f:
    DATA['inference_summary'] = json.load(f)

DATA['lambda_samples'] = np.load(CYCLE / 'infer' / 'lambda_samples.npy')

with open(CYCLE / 'results' / 'concordance.json') as f:
    DATA['concordance'] = json.load(f)

with open(CYCLE / 'results' / 'selection_bias.json') as f:
    DATA['selection_bias'] = json.load(f)

with open(CYCLE / 'results' / 'rank_comparison_report.md') as f:
    DATA['rank_comparison_md'] = f.read()

# Build lookup tables
ENTRY_NAMES = {e['entry_id']: e['canonical_name'] for e in DATA['rubric']['entries']}
INFER_ENTRY_ORDER = DATA['inference_summary']['entry_ids']

# Validate shapes
assert DATA['lambda_samples'].shape == (16000, 20), (
    f"Expected lambda_samples shape (16000, 20), got {DATA['lambda_samples'].shape}"
)
assert len(INFER_ENTRY_ORDER) == 20, (
    f"Expected 20 entry_ids in inference_summary, got {len(INFER_ENTRY_ORDER)}"
)

print(f"Loaded {len(DATA)} data files. {len(DATA['incidents'])} incidents, "
      f"{len(DATA['prelabels'])} prelabels, {len(DATA['goldset'])} adjudications, "
      f"{DATA['lambda_samples'].shape[0]} posterior draws. Ready.")

# What the Data Says About the 2026 Top 10

## Act 1: The Question

The OWASP Top 10 for LLMs ranks AI security vulnerabilities. The 2025 list was built
from expert surveys — hundreds of security professionals voting on what matters most.
That process produced a consensus: Prompt Injection at #1, Sensitive Information
Disclosure at #2, and so on down to #10.

Expert opinion is one signal. We wanted to check it against a second signal:
the pattern of real-world incidents. We built a corpus of ~6,600 AI security incidents
from public databases, classified each one against the 20-entry taxonomy, and asked:
does the incident data agree with the experts?

This notebook walks through that analysis step by step. Along the way, you will see
how the classification worked, how we measured its accuracy, and what a Bayesian model
does with noisy measurements. Every chart and table below is computed live from the
data — you can re-run any cell to verify.

Here are the 20 taxonomy entries we are working with. The "Incident Rank" column is
blank for now. We will fill it in Act 6, after walking through the methodology.

In [ ]:
# Build the entry table with a placeholder rank column
entry_table = pd.DataFrame([
    {'#': i + 1, 'Entry ID': e['entry_id'], 'Name': e['canonical_name'], 'Incident Rank': '—'}
    for i, e in enumerate(DATA['rubric']['entries'])
])
entry_table = entry_table.set_index('#')
display(entry_table.style.set_caption(
    "20 taxonomy entries. Incident Rank will be filled in Act 6."
).set_properties(**{'text-align': 'left'}))

## Act 2: The Corpus

The corpus contains 6,639 incidents from public databases: CVE, GHSA, and OSV
(security advisories), plus AIAAIC (a database of AI-related harms and controversies).
Each record has a text description of what happened.

The corpus splits into two strata. The **security** stratum (CVE/GHSA/OSV) contains
things like prompt injection exploits, data leakage through APIs, and supply chain
compromises in ML packages. The **ai-harm** stratum (AIAAIC) contains things like
algorithmic discrimination, deepfake misuse, and surveillance overreach.

This split matters. The classifier performs differently on each stratum — security
incidents have more structured descriptions (CVE format), while ai-harm incidents
are written as news summaries with varying detail.

In [ ]:
# Count incidents by stratum
inc_df = pd.DataFrame(DATA['incidents'])
stratum_counts = inc_df['stratum'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart by stratum
colors = ['#2c5282', '#c05621']
stratum_counts.plot.barh(ax=axes[0], color=colors[:len(stratum_counts)])
axes[0].set_title('Incidents by Stratum')
axes[0].set_xlabel('Number of incidents')
for i, (stratum, count) in enumerate(stratum_counts.items()):
    axes[0].text(count + 30, i, f'{count:,}', va='center', fontsize=11)

# Bar chart by entry (top 15)
entry_counts = inc_df[inc_df['entry_id'] != 'out-of-scope']['entry_id'].value_counts().head(15)
bar_colors = [ENTRY_COLORS.get(eid, '#666666') for eid in entry_counts.index]
entry_counts.plot.barh(ax=axes[1], color=bar_colors)
axes[1].set_title('Top 15 Entries by Incident Count')
axes[1].set_xlabel('Number of incidents')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Show 2 real incident examples — one from each stratum
# Build stratum lookup from classified incidents (prelabels has no stratum field)
_stratum_lookup = {inc['incident_id']: inc['stratum'] for inc in DATA['incidents']}

prelabels_df = DATA['prelabels']
security_ex = next((p for p in prelabels_df if p['triage_tier'] == 'agree'
                    and p['consensus'] not in ('out-of-scope', None)
                    and _stratum_lookup.get(p['incident_id']) == 'security'), None)
harm_ex = next((p for p in prelabels_df if p['triage_tier'] == 'agree'
                and _stratum_lookup.get(p['incident_id']) == 'ai-harm'
                and len(p['text']) > 100), None)
if security_ex is None or harm_ex is None:
    display(HTML('<p style="color: red;">Could not find suitable example incidents.</p>'))

def incident_card(record, label):
    """Format an incident as an HTML card."""
    text = html.escape(shorten(record['text'], width=250, placeholder='...'))
    consensus = html.escape(str(record['consensus']))
    tier = html.escape(str(record['triage_tier']))
    inc_id = html.escape(str(record['incident_id']))
    return HTML(
        f'<div style="border: 1px solid #ccc; border-radius: 8px; padding: 1em; '
        f'margin: 0.5em 0; background: #fafafa;">'
        f'<strong>{html.escape(label)}</strong><br>'
        f'<code>{inc_id}</code> · consensus: '
        f'<strong>{consensus}</strong> · tier: {tier}<br>'
        f'<p style="margin-top: 0.5em; color: #333;">{text}</p>'
        f'</div>'
    )

display(HTML('<h4>Example: Security stratum (CVE/GHSA/OSV)</h4>'))
display(incident_card(security_ex, 'Security incident'))

display(HTML('<h4>Example: AI-harm stratum (AIAAIC)</h4>'))
display(incident_card(harm_ex, 'AI-harm incident'))

In [ ]:
# F-frame sidebar
display(sidebar(
    'Deep dive: What the corpus cannot see (F-frame)',
    '<p>The corpus is built from a keyword crawl of public databases — CVE, GHSA, OSV, '
    'and AIAAIC. Incidents that never became CVEs or harm-database entries are invisible '
    'to us. A prompt injection attack against an internal enterprise tool that was caught '
    'and patched quietly will never appear in this data.</p>'
    '<p>This creates structural bias toward vulnerability types that get reported in '
    'public channels. Well-resourced organizations that fix issues internally are '
    'underrepresented. Novel attack types that have not been assigned a CVE category '
    'are invisible.</p>'
))

# F-circ callout — always visible, not collapsed
display(callout_box(
    '<strong>Structural limitation: taxonomy-frame circularity (F-circ)</strong><br><br>'
    'We classified these incidents using the same taxonomy we are trying to validate. '
    'If the classifier systematically favors certain entries, the incident counts will '
    'appear to confirm the expert rankings even if the true pattern is different. '
    'This is taxonomy-frame circularity. It means the concordance we measure later '
    'is an upper bound on true agreement, not a precise estimate of it.',
    border_color='#d69e2e', bg_color='#fefcbf'
))